# Pandas CSV und Excel: Streuungsmaße, Quantile und Dateien
### Unterrichtsmaterial | Julia Brutskaya

---
**Kontext:** Kurs-Umfrage der Datenwald-Akademie<br>
**Bibliotheken:** `pandas`, `numpy`, `openpyxl`

<img src="herr_bauchgefuehl.png" alt="Herr Bauchgefühl" width="170">

Heute Vormittag hast du Spannweite, Quartile, Quartilsabstand und Standardabweichung von Hand berechnet. Jetzt rechnest du dieselben Werte mit Pandas nach. Danach lernst du, wie Daten aus CSV- und Excel-Dateien in einen DataFrame kommen und wieder hinaus.

Herr Bauchgefühl ist wieder dabei, und der Ausreißer springt heute über einen Zaun.

| Block | Inhalt |
|---|---|
| A | Spannweite und Quartile |
| B | Quartilsabstand und Ausreißerregel |
| C | Varianz und Standardabweichung |
| D | CSV-Dateien einlesen und schreiben |
| E | Excel-Dateien einlesen und schreiben |
| F | Weitere Übungen |

---
# Block A – Spannweite und Quartile

## 1 – Die Daten laden

Du arbeitest mit der Kurs-Umfrage und mit zwei kleinen Kursen aus dem Skript von heute Vormittag. NumPy wird in Block C gebraucht.

Falls `openpyxl` (Py-Bibliothek, die es ermöglicht, Exel-Dateien zu lesen und in sie zu schreiben) noch fehlt, führe einmalig im Terminal aus (nötig für Excel-Dateien in Block E):

```bash
pip install openpyxl
```


In [1]:
import pandas as pd
import numpy as np

# Kurs-Umfrage der Datenwald-Akademie: eine Zeile pro befragter Person
zeilen = [
    (101, "Vormittag", "Kaffee", "M", 0, 25), (102, "Abend", "Tee", "S", 1, 35),
    (103, "Vormittag", "Mate", "L", 0, 18), (104, "Vormittag", "Kaffee", "L", 2, 42),
    (105, "Abend", "Wasser", "M", 0, 30), (106, "Vormittag", "Tee", "M", 1, 28),
    (107, "Abend", "Kaffee", "XL", 0, 55), (108, "Vormittag", "Kaffee", "S", 3, 12),
    (109, "Vormittag", "Mate", "M", 1, 38), (110, "Abend", "Tee", "M", 0, 115),
    (111, "Vormittag", "Kaffee", "L", 0, 32), (112, "Abend", "Mate", "L", 2, 45),
    (113, "Vormittag", "Wasser", "S", 1, 22), (114, "Vormittag", "Kaffee", "M", 0, 35),
    (115, "Abend", "Tee", "L", 1, 48), (116, "Vormittag", "Tee", "M", 0, 30),
    (117, "Abend", "Kaffee", "M", 3, 65), (118, "Vormittag", "Mate", "XL", 1, 40),
    (119, "Abend", "Wasser", "S", 0, 52), (120, "Vormittag", "Kaffee", "L", 2, 25),
]
spalten = ["teilnehmer_id", "kurszeit", "getraenk", "tshirt", "haustiere", "anfahrt_min"]
umfrage = pd.DataFrame(zeilen, columns=spalten)

# Abkürzung: die Spalte, mit der du heute am meisten rechnest
anfahrt = umfrage["anfahrt_min"]

# Zwei Mini-Kurse aus dem Skript: Anfahrt in Minuten
kurs_a = pd.Series([38, 39, 40, 41, 42])
kurs_b = pd.Series([5, 15, 25, 40, 115])

umfrage.head()

,teilnehmer_id,kurszeit,getraenk,tshirt,haustiere,anfahrt_min
0,101,Vormittag,Kaffee,M,0,25
1,102,Abend,Tee,S,1,35
2,103,Vormittag,Mate,L,0,18
3,104,Vormittag,Kaffee,L,2,42
4,105,Abend,Wasser,M,0,30


## 2 – Gleicher Mittelwert, andere Streuung

> **Herr Bauchgefühl:** „Kurs A und Kurs B brauchen beide im Schnitt 40 Minuten. Also sind die beiden Kurse gleich."

In [2]:
print("Kurs A:", kurs_a.tolist(), "| Mittelwert:", kurs_a.mean())
print("Kurs B:", kurs_b.tolist(), "| Mittelwert:", kurs_b.mean())

Kurs A: [38, 39, 40, 41, 42] | Mittelwert: 40.0
Kurs B: [5, 15, 25, 40, 115] | Mittelwert: 40.0


Derselbe Mittelwert, aber in Kurs A wohnen alle fast gleich weit weg, in Kurs B reicht die Anfahrt von 5 bis 115 Minuten. Ein Lagemaß allein reicht nicht. Wie stark die Werte schwanken, misst ein **Streuungsmaß**.

## 3 – Die Spannweite

Die Spannweite ist der Abstand zwischen dem größten und dem kleinsten Wert. Pandas hat dafür keine eigene Methode, du rechnest `max()` minus `min()`.

In [3]:
print("Spannweite Kurs A:", kurs_a.max() - kurs_a.min())
print("Spannweite Kurs B:", kurs_b.max() - kurs_b.min())

Spannweite Kurs A: 4
Spannweite Kurs B: 110


In [4]:
print("Spannweite Anfahrt (alle 20):", anfahrt.max() - anfahrt.min())

ohne = anfahrt[anfahrt < 100]
print("Spannweite ohne Ausreißer:  ", ohne.max() - ohne.min())

Spannweite Anfahrt (alle 20): 103
Spannweite ohne Ausreißer:   53


Ein einziger Wert halbiert die Spannweite fast: von 103 auf 53 Minuten. Die Spannweite hängt nur am größten und am kleinsten Wert und ist deshalb sehr empfindlich.

## 4 – Quartile mit `quantile()`

<img src="drilling_median.png" alt="Median" width="150">

Die Quartile teilen die sortierte Reihe in vier gleich große Teile. Der Median ist das zweite Quartil. In Pandas liefert `.quantile()` jedes beliebige Quantil: 0.25 für Q1, 0.5 für den Median, 0.75 für Q3.

In [5]:
print("Q1:    ", anfahrt.quantile(0.25))
print("Median:", anfahrt.quantile(0.5))
print("Q3:    ", anfahrt.quantile(0.75))

Q1:     27.25
Median: 35.0
Q3:     45.75


Mit einer Liste bekommst du mehrere Quantile auf einmal.

In [6]:
print(anfahrt.quantile([0.25, 0.5, 0.75]).tolist())

[27.25, 35.0, 45.75]


**Achtung, andere Rechenregel:** Von Hand hast du Q1 = 26,5 und Q3 = 46,5 berechnet, Pandas liefert 27,25 und 45,75. Beides ist richtig.

| | Q1 | Q3 | Regel |
|---|---|---|---|
| von Hand | 26,5 | 46,5 | Median der unteren und der oberen Hälfte |
| Pandas | 27,25 | 45,75 | rechnet anteilig zwischen zwei Nachbarwerten |

Für Quartile gibt es mehrere gebräuchliche Rechenregeln. Bei kleinen Datenreihen weichen sie leicht voneinander ab, bei großen fällt der Unterschied kaum auf.

Mit `interpolation="midpoint"` nimmt Pandas die Mitte zwischen den beiden Nachbarwerten. Bei diesen Daten trifft das genau deine Handwerte. Das gilt nicht für jede Datenreihe.

In [7]:
print(anfahrt.quantile([0.25, 0.75], interpolation="midpoint").tolist())

[26.5, 46.5]


## 5 – Die Fünf-Punkte-Zusammenfassung

Kleinster Wert, Q1, Median, Q3 und größter Wert zusammen heißen Fünf-Punkte-Zusammenfassung. `describe()` kennst du aus Notebook 2. Jetzt kannst du auch die Zeilen `25%`, `75%` und `std` lesen.

In [8]:
print(anfahrt.describe())

count     20.000000
mean      39.600000
std       22.020087
min       12.000000
25%       27.250000
50%       35.000000
75%       45.750000
max      115.000000
Name: anfahrt_min, dtype: float64


| Zeile | Bedeutung |
|---|---|
| `min`, `25%`, `50%`, `75%`, `max` | die Fünf-Punkte-Zusammenfassung (Quartile nach Pandas-Regel) |
| `std` | die Standardabweichung, Thema von Block C |

### Übung 1

- **a)** Berechne die Spannweite der Spalte `haustiere`.
- **b)** Berechne Q1, Median und Q3 der Anfahrt nur für den Vormittagskurs.
- **c)** Unter welcher Anfahrtszeit bleiben 90 % der Befragten? Berechne das 90-%-Quantil (das 90. Perzentil) und runde auf eine Nachkommastelle.

In [9]:
# a)


In [10]:
# b)


In [11]:
# c)


---
# Block B – Quartilsabstand und Ausreißerregel

## 6 – Der Quartilsabstand

Der **Quartilsabstand** (IQR) ist Q3 minus Q1. Er gibt an, wie breit die mittlere Hälfte der Daten ist. Die Ränder spielen keine Rolle, deshalb ist er robust.

In [12]:
# mit den Handwerten vom Vormittag
q1_hand, q3_hand = 26.5, 46.5
print("IQR von Hand:", q3_hand - q1_hand)

# mit den Pandas-Quartilen
q1 = anfahrt.quantile(0.25)
q3 = anfahrt.quantile(0.75)
iqr = q3 - q1
print("IQR mit Pandas:", iqr)

IQR von Hand: 20.0
IQR mit Pandas: 18.5


## 7 – Die Zäune: Wer ist ein Ausreißer?

<img src="der_ausreisser.png" alt="Der Ausreißer, der Kater" width="170">

> **Herr Bauchgefühl:** „Ausreißer erkenne ich mit bloßem Auge. Dafür brauche ich keine Regel."

Bei 20 Werten mag das klappen, bei 20.000 nicht. Die Regel vom Vormittag: Ein Wert ist ein Ausreißer, wenn er mehr als 1,5 Quartilsabstände unter Q1 oder über Q3 liegt. Diese beiden Grenzen heißen **Zäune**.

In [13]:
# Zäune mit den Handwerten
iqr_hand = q3_hand - q1_hand
unterer_zaun_hand = q1_hand - 1.5 * iqr_hand
oberer_zaun_hand = q3_hand + 1.5 * iqr_hand

print("Zäune von Hand:", unterer_zaun_hand, "und", oberer_zaun_hand)

Zäune von Hand: -3.5 und 76.5


In [14]:
# Zäune mit den Pandas-Quartilen
unterer_zaun = q1 - 1.5 * iqr
oberer_zaun = q3 + 1.5 * iqr

print("Zäune mit Pandas:", unterer_zaun, "und", oberer_zaun)

Zäune mit Pandas: -0.5 und 73.5


Die Grenzen unterscheiden sich leicht (76,5 und 73,5 Minuten), weil die Quartile verschieden berechnet sind. Entscheidend ist, wer außerhalb liegt. Das findest du mit einem Filter und dem Oder-Zeichen `|`.

In [15]:
ausreisser = umfrage[(anfahrt < unterer_zaun) | (anfahrt > oberer_zaun)]
ausreisser

,teilnehmer_id,kurszeit,getraenk,tshirt,haustiere,anfahrt_min
9,110,Abend,Tee,M,0,115


Nach beiden Rechenregeln springt genau ein Wert über den Zaun: die 115 Minuten. Gestern hieß es noch „liegt weit weg vom Rest", heute gibt es dafür eine Rechenregel.

### Übung 2

- **a)** Berechne den Quartilsabstand der Spalte `haustiere`.
- **b)** Gibt es im Vormittagskurs einen Ausreißer bei der Anfahrt? Berechne die Zäune nur für den Vormittagskurs und filtere.
- **c)** Wie viele der 20 Befragten liegen mit ihrer Anfahrt zwischen Q1 und Q3 (Grenzen eingeschlossen)?

In [16]:
# a)


In [17]:
# b)


In [18]:
# c)


---
# Block C – Varianz und Standardabweichung

## 8 – Von Hand nachvollziehen

<img src="drilling_mittelwert.png" alt="Mittelwert" width="150">

Die Standardabweichung misst, wie weit die Werte im Schnitt vom Mittelwert entfernt sind. Das Rezept vom Vormittag: Abweichungen vom Mittelwert bilden, quadrieren, mitteln, Wurzel ziehen. Pandas rechnet dabei mit der ganzen Spalte auf einmal.

In [19]:
abweichung = kurs_a - kurs_a.mean()

print("Abweichungen:        ", abweichung.tolist())
print("Summe der Abweichungen:", abweichung.sum())
print("Quadrate:            ", (abweichung ** 2).tolist())
print("Summe der Quadrate:  ", (abweichung ** 2).sum())

Abweichungen:         [-2.0, -1.0, 0.0, 1.0, 2.0]
Summe der Abweichungen: 0.0
Quadrate:             [4.0, 1.0, 0.0, 1.0, 4.0]
Summe der Quadrate:   10.0


Die Abweichungen heben sich gegenseitig auf, ihre Summe ist immer 0. Deshalb werden sie quadriert.

In [20]:
varianz = (abweichung ** 2).sum() / len(kurs_a)
standardabweichung = varianz ** 0.5

print("Varianz:           ", varianz)
print("Standardabweichung:", round(standardabweichung, 1))

Varianz:            2.0
Standardabweichung: 1.4


## 9 – `var()` und `std()`: teilen durch n oder durch n − 1?

> **Herr Bauchgefühl:** „Ob ich durch n oder durch n − 1 teile, ist doch egal."

Pandas hat für beides fertige Methoden: `.var()` für die Varianz, `.std()` für die Standardabweichung. Der Parameter `ddof` legt fest, wodurch geteilt wird.

| Daten | geteilt wird durch | in Pandas |
|---|---|---|
| vollständige Grundgesamtheit | n | `ddof=0` |
| Stichprobe | n − 1 | `ddof=1`, das ist der Standard |

Kurs A und Kurs B sind vollständig erfasst, also `ddof=0`.

In [21]:
print("Kurs A:", kurs_a.var(ddof=0), "|", round(kurs_a.std(ddof=0), 1))
print("Kurs B:", kurs_b.var(ddof=0), "|", round(kurs_b.std(ddof=0), 1))

Kurs A: 2.0 | 1.4
Kurs B: 1540.0 | 39.2


Gleicher Mittelwert, aber Kurs B streut rund 28-mal so stark wie Kurs A (39,2 gegen 1,4 Minuten).

Die Umfrage ist eine Stichprobe aus allen Teilnehmenden der Akademie. Hier passt der Pandas-Standard, der durch n − 1 teilt.

In [22]:
print("Stichprobe (n - 1):  ", round(anfahrt.var(), 1), "|", round(anfahrt.std(), 1))
print("Grundgesamtheit (n): ", round(anfahrt.var(ddof=0), 1), "|", round(anfahrt.std(ddof=0), 1))

Stichprobe (n - 1):   484.9 | 22.0
Grundgesamtheit (n):  460.6 | 21.5


22,0 gegen 21,5 Minuten: Egal ist es nicht, aber der Unterschied ist klein. Je größer die Stichprobe, desto kleiner wird er.

## 10 – Die Stolperfalle: NumPy teilt durch n

NumPy kennst du aus der Mathe-Woche. `np.std()` und `np.var()` teilen von sich aus durch **n**, Pandas durch **n − 1**. Dieselben Daten liefern also zwei verschiedene Ergebnisse, je nachdem, welche Bibliothek rechnet.

In [23]:
print("Pandas, anfahrt.std():         ", round(anfahrt.std(), 1))
print("NumPy, np.std(anfahrt):        ", round(np.std(anfahrt), 1))
print("NumPy, np.std(anfahrt, ddof=1):", round(np.std(anfahrt, ddof=1), 1))

Pandas, anfahrt.std():          22.0
NumPy, np.std(anfahrt):         21.5
NumPy, np.std(anfahrt, ddof=1): 22.0


| Aufruf | Standard | teilt durch |
|---|---|---|
| `spalte.std()`, `spalte.var()` (Pandas) | `ddof=1` | n − 1 |
| `np.std(...)`, `np.var(...)` (NumPy) | `ddof=0` | n |

Wenn zwei Auswertungen derselben Daten leicht verschiedene Standardabweichungen zeigen, liegt es fast immer daran.

## 11 – Robust oder empfindlich?

<img src="herr_bauchgefuehl.png" alt="Herr Bauchgefühl" width="140">

> **Herr Bauchgefühl:** „Die Standardabweichung klingt am wissenschaftlichsten. Die nehme ich ab jetzt immer."

Wie reagieren die Streuungsmaße, wenn der Ausreißer fehlt?

In [24]:
ohne = anfahrt[anfahrt < 100]

vergleich = pd.DataFrame({
    "mit Ausreißer": [
        anfahrt.max() - anfahrt.min(),
        anfahrt.quantile(0.75) - anfahrt.quantile(0.25),
        anfahrt.std(),
    ],
    "ohne Ausreißer": [
        ohne.max() - ohne.min(),
        ohne.quantile(0.75) - ohne.quantile(0.25),
        ohne.std(),
    ],
}, index=["Spannweite", "Quartilsabstand", "Standardabweichung"])

vergleich.round(1)

,mit Ausreißer,ohne Ausreißer
Spannweite,103.0,53.0
Quartilsabstand,18.5,17.0
Standardabweichung,22.0,13.4


Spannweite und Standardabweichung brechen ein, der Quartilsabstand ändert sich kaum. Daraus ergeben sich zwei Paare:

| Daten | Lagemaß | Streuungsmaß |
|---|---|---|
| symmetrisch, ohne Ausreißer | Mittelwert | Standardabweichung |
| schief oder mit Ausreißern | Median | Quartilsabstand |

Mit `groupby` und `agg` aus Notebook 2 vergleichst du die beiden Kurse in einem Schritt.

In [25]:
umfrage.groupby("kurszeit")["anfahrt_min"].agg(["mean", "median", "std", "min", "max"]).round(1)

,mean,median,std,min,max
kurszeit,,,,,
Abend,55.6,50.0,26.4,30,115
Vormittag,28.9,29.0,9.1,12,42


Der Abendkurs streut fast dreimal so stark wie der Vormittagskurs (26,4 gegen 9,1 Minuten). Ein großer Teil davon geht auf den einen Ausreißer zurück.

### Übung 3

- **a)** Berechne Varianz und Standardabweichung der Spalte `haustiere` (als Stichprobe). Runde auf zwei Nachkommastellen.
- **b)** Berechne die Standardabweichung der Anfahrt für jedes Getränk. Welche Gruppe streut am stärksten, und warum?
- **c)** Berechne die Standardabweichung von `haustiere` einmal mit Pandas und einmal mit `np.std()`. Erkläre den Unterschied als Kommentar.

In [26]:
# a)


In [27]:
# b)


In [28]:
# c)


---
# Block D – CSV-Dateien einlesen und schreiben

## 12 – Einen DataFrame als CSV speichern und wieder einlesen

Bisher standen die Daten immer im Notebook. In der Praxis kommen sie aus Dateien. Das häufigste Format ist **CSV** (*comma-separated values*): eine Textdatei mit einer Zeile pro Datensatz, die Werte durch ein Trennzeichen getrennt.

`.to_csv()` schreibt einen DataFrame in eine Datei. `index=False` verhindert, dass der Index (0, 1, 2, ...) als zusätzliche Spalte mitgeschrieben wird.

In [29]:
umfrage.to_csv("umfrage.csv", index=False)
print("Datei umfrage.csv gespeichert.")

Datei umfrage.csv gespeichert.


Eine CSV-Datei ist reiner Text. So sehen die ersten Zeilen aus:

In [30]:
with open("umfrage.csv", encoding="utf-8") as datei:
    for zeile in datei.readlines()[:4]:
        print(zeile.strip())

teilnehmer_id,kurszeit,getraenk,tshirt,haustiere,anfahrt_min
101,Vormittag,Kaffee,M,0,25
102,Abend,Tee,S,1,35
103,Vormittag,Mate,L,0,18


Die erste Zeile enthält die Spaltennamen, die Werte sind durch Kommas getrennt. Genau das erwartet `pd.read_csv()` von sich aus.

In [31]:
eingelesen = pd.read_csv("umfrage.csv")
eingelesen.head(3)

,teilnehmer_id,kurszeit,getraenk,tshirt,haustiere,anfahrt_min
0,101,Vormittag,Kaffee,M,0,25
1,102,Abend,Tee,S,1,35
2,103,Vormittag,Mate,L,0,18


## 13 – Semikolon als Trennzeichen

<img src="herr_bauchgefuehl.png" alt="Herr Bauchgefühl" width="140">

> **Herr Bauchgefühl:** „Eine CSV-Datei ist eine CSV-Datei. Die sehen alle gleich aus."

Leider nicht. Ein deutsch eingestelltes Excel trennt die Werte mit einem **Semikolon**. Mit dem Parameter `sep` legst du das Trennzeichen fest, beim Schreiben wie beim Lesen.

In [32]:
umfrage.to_csv("umfrage_semikolon.csv", sep=";", index=False)

with open("umfrage_semikolon.csv", encoding="utf-8") as datei:
    for zeile in datei.readlines()[:3]:
        print(zeile.strip())

teilnehmer_id;kurszeit;getraenk;tshirt;haustiere;anfahrt_min
101;Vormittag;Kaffee;M;0;25
102;Abend;Tee;S;1;35


Liest du diese Datei ohne Angabe ein, findet Pandas kein Komma und packt jede Zeile in eine einzige Spalte.

In [33]:
falsch = pd.read_csv("umfrage_semikolon.csv")
print("Zeilen und Spalten:", falsch.shape)
falsch.head(2)

Zeilen und Spalten: (20, 1)


,teilnehmer_id;kurszeit;getraenk;tshirt;haustiere;anfahrt_min
0,101;Vormittag;Kaffee;M;0;25
1,102;Abend;Tee;S;1;35


In [34]:
richtig = pd.read_csv("umfrage_semikolon.csv", sep=";")
print("Zeilen und Spalten:", richtig.shape)
richtig.head(2)

Zeilen und Spalten: (20, 6)


,teilnehmer_id,kurszeit,getraenk,tshirt,haustiere,anfahrt_min
0,101,Vormittag,Kaffee,M,0,25
1,102,Abend,Tee,S,1,35


**Merke:** Nach jedem Einlesen `shape` und `head()` prüfen. Eine einzige Spalte mit langen Texten ist fast immer ein falsches Trennzeichen.

## 14 – Dezimalkomma

In deutschen Dateien steht 0,42 statt 0.42. Die folgende Datei enthält die Anfahrt in Stunden, geschrieben mit Semikolon und Dezimalkomma.

In [35]:
stunden = pd.DataFrame({
    "teilnehmer_id": umfrage["teilnehmer_id"],
    "anfahrt_std": (umfrage["anfahrt_min"] / 60).round(2),
})
stunden.to_csv("anfahrt_stunden.csv", sep=";", decimal=",", index=False)

with open("anfahrt_stunden.csv", encoding="utf-8") as datei:
    for zeile in datei.readlines()[:3]:
        print(zeile.strip())

teilnehmer_id;anfahrt_std
101;0,42
102;0,58


Mit `sep=";"` allein stimmen die Spalten, aber Pandas hält `0,42` für Text. Mit Text kannst du nicht rechnen.

In [36]:
als_text = pd.read_csv("anfahrt_stunden.csv", sep=";")
print(als_text.dtypes)

teilnehmer_id    int64
anfahrt_std        str
dtype: object


Erst `decimal=","` macht aus der Spalte Zahlen.

In [37]:
als_zahl = pd.read_csv("anfahrt_stunden.csv", sep=";", decimal=",")

print(als_zahl.dtypes)
print()
print("Mittlere Anfahrt:", round(als_zahl["anfahrt_std"].mean(), 2), "Stunden")

teilnehmer_id      int64
anfahrt_std      float64
dtype: object

Mittlere Anfahrt: 0.66 Stunden


**Merke:** Nach dem Einlesen auch `dtypes` prüfen. Eine Zahlenspalte, die als Text angezeigt wird, ist ein Warnsignal.

## 15 – Kommentarzeilen und fehlende Kopfzeile

Exporte aus Verwaltungssystemen beginnen oft mit Kommentaren und haben manchmal keine Zeile mit Spaltennamen.

In [38]:
inhalt = (
    "# Export: Datenwald-Akademie\n"
    "# Erstellt am: 2026-10-07\n"
    "101,Vormittag,25\n"
    "102,Abend,35\n"
    "103,Vormittag,18\n"
)

with open("anfahrt_export.csv", "w", encoding="utf-8") as datei:
    datei.write(inhalt)

print(inhalt)

# Export: Datenwald-Akademie
# Erstellt am: 2026-10-07
101,Vormittag,25
102,Abend,35
103,Vormittag,18



Drei Parameter lösen das:

| Parameter | Wirkung |
|---|---|
| `skiprows=2` | überspringt die ersten zwei Zeilen |
| `header=None` | die Datei hat keine Kopfzeile |
| `names=[...]` | vergibt eigene Spaltennamen |

In [39]:
export = pd.read_csv(
    "anfahrt_export.csv",
    skiprows=2,
    header=None,
    names=["teilnehmer_id", "kurszeit", "anfahrt_min"],
)
export

,teilnehmer_id,kurszeit,anfahrt_min
0,101,Vormittag,25
1,102,Abend,35
2,103,Vormittag,18


## 16 – Umlaute und Encoding

Jede Textdatei ist in einer Zeichenkodierung gespeichert. Heute üblich ist `utf-8`, und das nimmt Pandas von sich aus an. Dateien aus älteren Windows-Programmen sind oft in `latin1` gespeichert. Dann scheitert das Einlesen an den Umlauten.

In [40]:
# Eine Datei mit Umlauten in der alten Kodierung latin1 schreiben
with open("getraenke_alt.csv", "w", encoding="latin1") as datei:
    datei.write("Getränk;Größe\nKaffee;groß\nTee;klein\n")

try:
    pd.read_csv("getraenke_alt.csv", sep=";")
except UnicodeDecodeError:
    print("Fehler: Die Datei ist nicht in utf-8 gespeichert.")

Fehler: Die Datei ist nicht in utf-8 gespeichert.


In [41]:
# Mit der richtigen Kodierung klappt es
pd.read_csv("getraenke_alt.csv", sep=";", encoding="latin1")

,Getränk,Größe
0,Kaffee,groß
1,Tee,klein


**Zusammengefasst:** die wichtigsten Parameter von `read_csv`

| Problem | Parameter |
|---|---|
| anderes Trennzeichen | `sep=";"`, bei Tabulatoren `sep="\t"` |
| Dezimalkomma | `decimal=","` |
| Zeilen vor der Tabelle | `skiprows=2` |
| keine Kopfzeile | `header=None`, `names=[...]` |
| Umlaute kaputt oder Fehler beim Lesen | `encoding="latin1"` |

Jetzt weißt du auch, was hinter der vorgegebenen Zeile aus Notebook 1 und 2 steckt: Die Charts-Datei trennt ihre Spalten mit Tabulatoren, deshalb `sep="\t"`.

### Übung 4

Die Akademie hat die Auslastung ihrer Räume exportiert. Führe zuerst die Zelle aus, die die Datei `raumbelegung.csv` erstellt.

- **a)** Lies die Datei so ein, dass ein DataFrame mit drei sauberen Spalten entsteht.
- **b)** Prüfe mit `dtypes`, ob `Plaetze` und `Auslastung_pct` als Zahlen erkannt wurden.
- **c)** Berechne Mittelwert und Standardabweichung der Auslastung. Runde auf eine Nachkommastelle.

In [42]:
with open("raumbelegung.csv", "w", encoding="utf-8") as datei:
    datei.write("# Raumbelegung Datenwald-Akademie\n")
    datei.write("Raum;Plaetze;Auslastung_pct\n")
    datei.write("Eiche;24;87,5\n")
    datei.write("Buche;16;62,5\n")
    datei.write("Linde;30;93,3\n")
    datei.write("Ahorn;12;41,7\n")

print("Datei raumbelegung.csv erstellt.")

Datei raumbelegung.csv erstellt.


In [43]:
# a)


In [44]:
# b)


In [45]:
# c)


---
# Block E – Excel-Dateien einlesen und schreiben

## 17 – Einen DataFrame als Excel-Datei speichern und einlesen

Für Excel gibt es dieselben beiden Richtungen: `.to_excel()` schreibt, `pd.read_excel()` liest. Im Hintergrund arbeitet die Bibliothek `openpyxl`.

Anders als bei CSV gibt es bei Excel kein Trennzeichen und kein Dezimalkomma zu beachten. Zahlen bleiben Zahlen.

In [46]:
umfrage.to_excel("umfrage.xlsx", index=False)

aus_excel = pd.read_excel("umfrage.xlsx")
aus_excel.head(3)

,teilnehmer_id,kurszeit,getraenk,tshirt,haustiere,anfahrt_min
0,101,Vormittag,Kaffee,M,0,25
1,102,Abend,Tee,S,1,35
2,103,Vormittag,Mate,L,0,18


## 18 – Mehrere Tabellenblätter

Eine Excel-Datei kann mehrere Tabellenblätter enthalten. Zum Schreiben öffnest du die Datei mit `pd.ExcelWriter` und gibst jedem DataFrame mit `sheet_name` ein eigenes Blatt.

In [47]:
vormittagskurs = umfrage[umfrage["kurszeit"] == "Vormittag"]
abendkurs = umfrage[umfrage["kurszeit"] == "Abend"]

with pd.ExcelWriter("umfrage_kurse.xlsx") as writer:
    vormittagskurs.to_excel(writer, sheet_name="Vormittag", index=False)
    abendkurs.to_excel(writer, sheet_name="Abend", index=False)

print("Datei umfrage_kurse.xlsx mit zwei Blättern gespeichert.")

Datei umfrage_kurse.xlsx mit zwei Blättern gespeichert.


Ohne Angabe liest `read_excel` nur das **erste** Blatt. Mit `sheet_name` wählst du ein bestimmtes.

In [48]:
erstes_blatt = pd.read_excel("umfrage_kurse.xlsx")
print("Ohne Angabe:", erstes_blatt["kurszeit"].unique().tolist(), "|", len(erstes_blatt), "Zeilen")

abend_blatt = pd.read_excel("umfrage_kurse.xlsx", sheet_name="Abend")
print("Blatt Abend:", abend_blatt["kurszeit"].unique().tolist(), "|", len(abend_blatt), "Zeilen")

Ohne Angabe: ['Vormittag'] | 12 Zeilen
Blatt Abend: ['Abend'] | 8 Zeilen


## 19 – Eine Auswertung exportieren

Meistens exportierst du nicht die Rohdaten, sondern ein Ergebnis. Bei einer Gruppenauswertung stehen die Gruppennamen im Index. Hier lässt du `index=False` weg, sonst fehlen sie in der Datei.

In [49]:
streuung = umfrage.groupby("kurszeit")["anfahrt_min"].agg(["mean", "median", "std"]).round(1)
streuung

,mean,median,std
kurszeit,,,
Abend,55.6,50.0,26.4
Vormittag,28.9,29.0,9.1


In [50]:
streuung.to_excel("streuung_bericht.xlsx", sheet_name="Streuung")

pd.read_excel("streuung_bericht.xlsx", sheet_name="Streuung")

,kurszeit,mean,median,std
0,Abend,55.6,50,26.4
1,Vormittag,28.9,29,9.1


| Richtung | CSV | Excel |
|---|---|---|
| lesen | `pd.read_csv("datei.csv")` | `pd.read_excel("datei.xlsx")` |
| schreiben | `df.to_csv("datei.csv", index=False)` | `df.to_excel("datei.xlsx", index=False)` |

### Übung 5

- **a)** Lies aus `umfrage_kurse.xlsx` das Blatt `Vormittag` ein und berechne Median und Quartilsabstand der Anfahrt.
- **b)** Erstelle eine Auswertung pro Getränk mit Anzahl, Mittelwert und Standardabweichung der Anfahrt, gerundet auf eine Nachkommastelle. Speichere sie als `getraenke_bericht.xlsx` im Blatt `Getraenke`.
- **c)** Lies die Datei aus b) wieder ein und zeige sie an. Sind die Getränkenamen enthalten?

In [51]:
# a)


In [52]:
# b)


In [53]:
# c)


---
# Block F – Weitere Übungen

Jede Übung nennt den Block, zu dem sie gehört.

- **Übung 6 bis 9:** Gegenprobe zu deinen Handrechnungen vom Vormittag.
- **Übung 10:** Bonus mit Variationskoeffizient und z-Wert.
- **Übung 11:** Die Charts.
- **Übung 12 bis 15:** Die TechAcademy GmbH.
- **Übung 16 bis 19:** Zusatz mit Excel-Sonderfällen und weiteren Dateien der TechAcademy.

## 20 – Gegenprobe zum Vormittag

Von Hand hast du die Quartile nach dem Rezept „Median der Hälften" berechnet. Bei diesen Übungsdaten trifft `interpolation="midpoint"` deine Handwerte, der Pandas-Standard weicht leicht ab.

### Übung 6 – Acht Prüfungsergebnisse (zu Block A)

Vormittags war das Übung 2. Berechne Q1, Median und Q3 der Punkte einmal mit `interpolation="midpoint"` und einmal mit dem Pandas-Standard.

In [54]:
punkte = pd.Series([70, 52, 84, 61, 93, 75, 64, 78])

In [55]:
# Übung 6


### Übung 7 – Der Abendkurs und sein Zaun (zu Block B)

Vormittags war das Übung 4. Berechne für die Anfahrt des Abendkurses Q1 und Q3 mit `interpolation="midpoint"`, daraus den Quartilsabstand und die beiden Zäune. Welcher Wert ist ein Ausreißer?

In [56]:
# Übung 7


### Übung 8 – Lernstunden (zu Block C)

Vormittags waren das Übung 5 und 7. Fünf Teilnehmende lernen pro Woche 4, 6, 5, 9 und 6 Stunden. Berechne Varianz und Standardabweichung einmal für eine Grundgesamtheit und einmal für eine Stichprobe.

In [57]:
lernen = pd.Series([4, 6, 5, 9, 6])

In [58]:
# Übung 8


### Übung 9 – Gehälter mit und ohne Chef (zu Block A und C)

Vormittags war das Übung 8. Berechne Spannweite und Quartilsabstand der Gehälter einmal mit Herrn Bauchgefühl (14.000 €) und einmal ohne. Welches Streuungsmaß reagiert stärker?

*Hinweis: `gehalt[:-1]` enthält alle Werte außer dem letzten.*

In [59]:
gehalt = pd.Series([2400, 2600, 2800, 3000, 3200, 14000])

In [60]:
# Übung 9


## 21 – Bonus: Variationskoeffizient und z-Wert

Der **Variationskoeffizient** teilt die Standardabweichung durch den Mittelwert. So lässt sich die Streuung von Merkmalen mit verschiedenen Einheiten vergleichen.

In [61]:
print("Anfahrt:  ", round(anfahrt.std() / anfahrt.mean(), 2))
print("Haustiere:", round(umfrage["haustiere"].std() / umfrage["haustiere"].mean(), 2))

Anfahrt:   0.56
Haustiere: 1.13


Die Zahl der Haustiere streut im Verhältnis zu ihrem Mittelwert doppelt so stark wie die Anfahrt.

Der **z-Wert** sagt, wie viele Standardabweichungen ein Wert vom Mittelwert entfernt ist. Pandas berechnet ihn für die ganze Spalte auf einmal.

In [62]:
umfrage["z_anfahrt"] = (anfahrt - anfahrt.mean()) / anfahrt.std()

umfrage.sort_values("z_anfahrt", ascending=False)[["teilnehmer_id", "anfahrt_min", "z_anfahrt"]].head(3).round(2)

,teilnehmer_id,anfahrt_min,z_anfahrt
9,110,115,3.42
16,117,65,1.15
6,107,55,0.70


Die 115 Minuten liegen 3,4 Standardabweichungen über dem Mittelwert. Schon der zweitgrößte Wert kommt nur auf 1,15.

### Übung 10 – Bonus (zu Block C)

- **a)** Berechne für jede Kurszeit den Variationskoeffizienten der Anfahrt. Welcher Kurs streut relativ stärker?
- **b)** Füge eine Spalte `z_haustiere` mit den z-Werten der Haustiere hinzu. Wie groß ist der höchste z-Wert?

In [63]:
# a)


In [64]:
# b)


## 22 – Die Charts

Die Datei `UK-top40-1964-1-2.tsv` muss im selben Ordner liegen wie dieses Notebook. Diesmal liest du sie selbst ein.

### Übung 11 – Wie unterschiedlich lange bleiben Songs in den Charts? (zu Block A bis D)

- **a)** Lies die Datei in einen DataFrame `charts` ein. Die Spalten sind durch Tabulatoren getrennt. Prüfe das Ergebnis mit `shape` und `head()`.
- **b)** Berechne für `WoC` (Wochen in den Charts) Spannweite, Q1, Q3, Quartilsabstand und Standardabweichung.
- **c)** Welche Songs sind nach der 1,5-IQR-Regel Ausreißer?
- **d)** Welches Paar aus Lage- und Streuungsmaß würdest du berichten, und warum? Antworte als Kommentar.

In [65]:
# a)


In [66]:
# b)


In [67]:
# c)


In [68]:
# d)


## 23 – Zweiter Schauplatz: die TechAcademy GmbH

Führe die folgende Zelle aus. Sie erstellt vier Dateien, die du in den Übungen einliest.

In [69]:
# Datei 1: CSV mit Semikolon und Dezimalkomma
with open("geraete_messung.csv", "w", encoding="utf-8") as datei:
    datei.write("GeraeteID;Temperatur;Spannung_V\n")
    datei.write("G01;36,8;220,5\n")
    datei.write("G02;41,2;219,0\n")
    datei.write("G03;38,5;221,3\n")
    datei.write("G04;39,1;218,7\n")

# Datei 2: CSV ohne Kopfzeile, mit zwei Kommentarzeilen
with open("zugangsprotokolle.csv", "w", encoding="utf-8") as datei:
    datei.write("# Zugangsprotokolle TechAcademy GmbH\n")
    datei.write("# Exportiert am: 2024-05-01\n")
    datei.write("T-201,login,2024-05-01 08:12:00\n")
    datei.write("T-204,kurs_gestartet,2024-05-01 08:15:30\n")
    datei.write("T-201,logout,2024-05-01 09:45:00\n")
    datei.write("T-207,login,2024-05-01 10:00:15\n")

# Datei 3: Excel mit zwei Tabellenblättern
halbjahr_1 = pd.DataFrame({
    "Kursname": ["Python Grundlagen", "SQL für Einsteiger", "Data Analysis"],
    "Anmeldungen": [68, 45, 57],
    "Einnahmen_EUR": [54332, 26955, 56943],
})
halbjahr_2 = pd.DataFrame({
    "Kursname": ["Python Grundlagen", "Linux Basics", "Power BI"],
    "Anmeldungen": [74, 31, 52],
    "Einnahmen_EUR": [59126, 15469, 46748],
})
with pd.ExcelWriter("halbjahresbericht.xlsx") as writer:
    halbjahr_1.to_excel(writer, sheet_name="H1", index=False)
    halbjahr_2.to_excel(writer, sheet_name="H2", index=False)

# Datei 4: Standard-CSV
with open("trainer.csv", "w", encoding="utf-8") as datei:
    datei.write("Name,Fachbereich,Stundenlohn_EUR\n")
    datei.write("Sandra Meier,Python & Data,55\n")
    datei.write("Tobias Lange,Datenbanken,60\n")
    datei.write("Yara Özcan,Linux & Netzwerk,48\n")
    datei.write("Michael Brandt,Python & Data,72\n")

print("Vier Dateien erstellt.")

Vier Dateien erstellt.


### Übung 12 – Messwerte mit Semikolon und Dezimalkomma (zu Block D und C)

- **a)** Lies `geraete_messung.csv` so ein, dass `Temperatur` und `Spannung_V` als Zahlen erkannt werden. Prüfe es mit `dtypes`.
- **b)** Berechne Mittelwert und Standardabweichung der Temperatur.

In [70]:
# a)


In [71]:
# b)


### Übung 13 – Protokoll ohne Kopfzeile (zu Block D)

Lies `zugangsprotokolle.csv` ein. Die ersten zwei Zeilen sind Kommentare, eine Kopfzeile gibt es nicht. Der DataFrame soll die Spalten `TeilnehmerID`, `Aktion` und `Zeitstempel` bekommen.

In [72]:
# Übung 13


### Übung 14 – Das zweite Halbjahr (zu Block E)

Lies aus `halbjahresbericht.xlsx` nur das Tabellenblatt `H2` ein und berechne die Summe der Spalte `Einnahmen_EUR`.

In [73]:
# Übung 14


### Übung 15 – Einlesen, bearbeiten, exportieren (zu Block D und E)

- **a)** Lies `trainer.csv` ein.
- **b)** Füge eine Spalte `Monatsgehalt_EUR` hinzu. Vereinfacht gilt: Stundenlohn mal 5 Tage mal 4 Wochen mal 6 Stunden.
- **c)** Speichere das Ergebnis als `trainer_report.xlsx` im Blatt `Auswertung`, ohne den Index.

In [74]:
# a)


In [75]:
# b)


In [76]:
# c)


## 24 – Zusatz: Excel-Sonderfälle

Excel-Dateien aus der Praxis sind selten so sauber wie die aus Block E. Vier Fälle kommen immer wieder vor. Die Beispiele stammen von der TechAcademy GmbH.

**Fall 1: Alle Tabellenblätter auf einmal.** Mit `sheet_name=None` liest Pandas alle Blätter und gibt ein Dictionary zurück: Der Blattname ist der Schlüssel, der DataFrame der Wert.

In [ ]:
# Die vorhandenen Dateien liegen in einem Unterordner neben dem Notebook
ordner = "csv & excel Dateien/"

alle_blaetter = pd.read_excel(ordner + "quartalsbericht.xlsx", sheet_name=None)

print("Vorhandene Blätter:", list(alle_blaetter.keys()))
alle_blaetter["Q2"]

**Fall 2: Titelzeilen über der Tabelle.** In `bericht_mit_titel.xlsx` stehen ein Titel und eine Leerzeile über den Spaltennamen. Ohne Angabe hält Pandas den Titel für die Kopfzeile.

In [ ]:
pd.read_excel(ordner + "bericht_mit_titel.xlsx").head(3)

`skiprows=2` überspringt die ersten zwei Zeilen. Die nächste Zeile wird dann zur Kopfzeile.

In [ ]:
pd.read_excel(ordner + "bericht_mit_titel.xlsx", skiprows=2)

**Fall 3: Nur bestimmte Spalten.** Bei breiten Tabellen lädst du mit `usecols` nur, was du brauchst: entweder über die Excel-Buchstaben oder über die Spaltennamen.

In [ ]:
print(pd.read_excel(ordner + "quartalsbericht.xlsx", usecols="A:B"))
print()
print(pd.read_excel(ordner + "quartalsbericht.xlsx", usecols=["Kursname", "Einnahmen_EUR"]))

**Fall 4: Eigene Kennzeichen für fehlende Werte.** In `kurse_fehlwerte.xlsx` steht statt einer leeren Zelle `k.A.` oder `-`. Pandas hält das für Text, und die Zahlenspalte ist keine Zahlenspalte mehr.

In [ ]:
roh = pd.read_excel(ordner + "kurse_fehlwerte.xlsx")

print(roh)
print()
print(roh.dtypes)

Mit `na_values` nennst du die Kennzeichen, die als fehlend gelten sollen. `N/A` erkennt Pandas von selbst.

In [ ]:
sauber = pd.read_excel(ordner + "kurse_fehlwerte.xlsx", na_values=["k.A.", "-"])

print(sauber)
print()
print("Fehlende Werte pro Spalte:")
print(sauber.isnull().sum())
print()
print("Mittlere Anmeldungen:", sauber["Anmeldungen"].mean())

### Übung 16 – Kursbericht mit Titelzeilen (zu Abschnitt 24)

Die Datei `kursbericht_q2.xlsx` enthält drei Zeilen vor der eigentlichen Tabelle.

- **a)** Lies die Datei so ein, dass die vierte Zeile als Kopfzeile verwendet wird.
- **b)** Berechne die Summe der Teilnehmerzahl und den Mittelwert der Bewertung.

In [ ]:
# a) 

In [ ]:
# b) 

## 25 – Sechs Dateien, sechs Probleme

Die TechAcademy hat ihre Kursauslastung aus sechs verschiedenen Systemen exportiert. Jede Datei hat eine andere Eigenheit. Führe zuerst die folgende Zelle aus, sie bereitet die sechste Datei vor.

In [ ]:
# Die sechste Datei: Umlaute in der alten Kodierung latin1
with open(ordner + "kurse_umlaute.csv", "w", encoding="latin1") as datei:
    datei.write("Kursname;Teilnehmerzahl\n")
    datei.write("Grundlagen Bürokommunikation;12\n")
    datei.write("Datenschutz für Einsteiger;9\n")

print("Datei kurse_umlaute.csv vorbereitet.")

### Übung 17 – Sechs Dateien richtig einlesen (zu Block D)

Die Dateien liegen im Unterordner `csv & excel Dateien`. Lies jede so ein, dass ein sauberer DataFrame entsteht. Prüfe jedes Ergebnis mit `shape`, `head()` und `dtypes`. Öffne die Datei im Zweifel zuerst im Editor und sieh nach, wie sie aufgebaut ist.

- **a)** `kursauslastung_standard.csv`
- **b)** `kursauslastung_semikolon.csv`
- **c)** `kursauslastung_kein_header.csv` (Spalten: `Kursname`, `Monat`, `Anmeldungen`, `Auslastung_pct`)
- **d)** `kursauslastung_mit_intro.csv`
- **e)** `kursauslastung_dezimal.csv`
- **f)** `kurse_umlaute.csv`
- **g)** Berechne für die Datei aus a) Mittelwert und Standardabweichung der Spalte `Auslastung_pct`.

*Hinweis: Den Ordner hängst du mit `ordner + "dateiname"` vor den Dateinamen.*

In [ ]:
# a)

In [ ]:
# b)

In [ ]:
# c)

In [ ]:
# d)

In [ ]:
# e)

In [ ]:
# f)

In [ ]:
# g)

## 26 – Zwei weitere Excel-Dateien

### Übung 18 – Klausurpunkte im Januar und Februar (zu Block C und E)

Die Datei `Klausurpunkte.xlsx` hat zwei Tabellenblätter: `Januar` und `Februar`.

- **a)** Lies beide Blätter in je einen DataFrame ein.
- **b)** Berechne für jeden Monat Mittelwert und Standardabweichung der Punkte. Runde auf zwei Nachkommastellen.
- **c)** In welchem Monat waren die Ergebnisse besser, in welchem lagen sie näher beieinander? Antworte als Kommentar.

In [ ]:
# a)

In [ ]:
# b)

In [ ]:
# c)

### Übung 19 – Der Verkaufsreport (zu Abschnitt 24)

Lies `report.xlsx` so ein, dass die Spalten `Datum`, `Produkt` und `Verkaufte Einheiten` heißen. Berechne danach die Summe der verkauften Einheiten.

*Hinweis: Lies die Datei zuerst ohne Parameter ein und sieh nach, in welcher Zeile die Spaltennamen stehen.*

In [ ]:
# Übung 19

---
## 27 – Zusammenfassung

| Aufgabe | Befehl |
|---|---|
| Spannweite | `spalte.max() - spalte.min()` |
| Quartile und Quantile | `quantile(0.25)`, `quantile([0.25, 0.5, 0.75])` |
| Quartile wie von Hand | `quantile(..., interpolation="midpoint")` |
| Quartilsabstand | `quantile(0.75) - quantile(0.25)` |
| Zäune | `q1 - 1.5 * iqr` und `q3 + 1.5 * iqr` |
| Varianz und Standardabweichung | `var()`, `std()`; Stichprobe `ddof=1` (Standard), Grundgesamtheit `ddof=0` |
| Achtung NumPy | `np.std()` teilt durch n |
| CSV lesen | `pd.read_csv(datei, sep=, decimal=, skiprows=, header=, names=, encoding=)` |
| CSV schreiben | `df.to_csv(datei, index=False)` |
| Excel lesen | `pd.read_excel(datei, sheet_name=)` |
| Excel schreiben | `df.to_excel(datei, sheet_name=, index=False)`, mehrere Blätter mit `pd.ExcelWriter` |
| Excel-Sonderfälle | `sheet_name=None`, `skiprows=`, `usecols=`, `na_values=` |

**Nach jedem Einlesen prüfen:** `shape`, `head()` und `dtypes`.

---
*Material erstellt von Julia Brutskaya*